### Building a RAG System with LangChain and FAISS
Introduction to RAG (Retrieval-Augmented Generation)
RAG combines the power of retrieval systems with generative AI models. Instead of relying solely on the model's training data, RAG:

1. Retrieves relevant documents from a knowledge base
2. Uses these documents as context for the LLM
3. Generates responses based on both the retrieved context and the model's knowledge

### FAISS
https://github.com/facebookresearch/faiss

FAISS is a library for efficient similarity search and clustering of dense vectors.

Key advantages:
1. Extremely fast similarity search
2. Memory efficient
3. Supports GPU acceleration
4. Can handle millions of vectors

How it works:
- Indexes vectors for fast nearest neighbor search
- Returns most similar vectors based on distance metrics


In [55]:
# !pip install langchain_google_genai langchain_groq langchain_community langchain

In [56]:
## load libraries
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# LangChain core imports
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import (
    RunnablePassthrough,

)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
import os

# LangChain specific imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

### Data Ingestion And Processing


In [57]:
sample_documents = [
    Document(
        page_content="""
        Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
        """,
        metadata={"source": "AI Introduction", "page": 1, "topic": "AI"}
    ),
    Document(
        page_content="""
        Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.
        """,
        metadata={"source": "ML Basics", "page": 1, "topic": "ML"}
    ),
    Document(
        page_content="""
        Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning has revolutionized computer vision, NLP, and speech recognition.
        """,
        metadata={"source": "Deep Learning", "page": 1, "topic": "DL"}
    ),
    Document(
        page_content="""
        Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
        Applications include chatbots, translation, sentiment analysis, and text summarization.
        """,
        metadata={"source": "NLP Overview", "page": 1, "topic": "NLP"}
    )
]

print(sample_documents)

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='\n        Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.\n        '), Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='\n        Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.\n        '), Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='\n        Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolu

In [58]:
## text splitting
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=[" "]
)

## split the documents into chunks
chunks = text_splitter.split_documents(sample_documents)
print(chunks[0])
print(chunks[1])


page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.' metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}
page_content='Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised, unsupervised, and reinforcement learning.' metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}


In [59]:
print(f"Created {len(chunks)} chunks from {len(sample_documents)} documents")
print("\nExample chunk:")
print(f"Content: {chunks[0].page_content}")
print(f"Metadata: {chunks[0].metadata}")

Created 4 chunks from 4 documents

Example chunk:
Content: Artificial Intelligence (AI) is the simulation of human intelligence in machines.
        These systems are designed to think like humans and mimic their actions.
        AI can be categorized into narrow AI and general AI.
Metadata: {'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}


In [60]:
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [61]:
# Initialize OpenAI embeddings with the latest model
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2",
)

## Example: create a embedding for a single text
sample_text="What is machine learning"
sample_embedding=embeddings.embed_query(sample_text)
sample_embedding

[-0.016195752,
 -0.0010386702,
 0.006682801,
 0.02343683,
 0.0063548936,
 0.003344212,
 -0.020864505,
 0.0027927822,
 0.022513947,
 -0.05925505,
 -0.012252547,
 0.0065319315,
 -0.024448054,
 -0.005638863,
 0.0009440519,
 -0.004130744,
 0.0448177,
 -0.004566203,
 0.0066051106,
 -0.006798212,
 -0.0032384428,
 0.0144693125,
 0.010936454,
 0.011823718,
 -0.005612606,
 0.025238752,
 -0.022829445,
 -0.0028819256,
 -0.009461555,
 0.13059051,
 -0.020959737,
 -0.040670644,
 -0.008995341,
 -0.0070034107,
 -0.005696987,
 0.009820383,
 -0.0064979363,
 -0.0034067424,
 -0.002674538,
 -0.008625696,
 0.0026962168,
 0.00707075,
 0.016127711,
 0.010465013,
 -0.0047193663,
 -0.019445559,
 -0.023117853,
 -8.446642e-05,
 0.007676487,
 -0.0036722252,
 -0.0046645734,
 0.017272223,
 -0.011026724,
 -0.027329747,
 -0.00024222374,
 -0.011902998,
 0.018416734,
 0.019346341,
 0.019991113,
 -0.00496173,
 -0.0050915815,
 0.01846233,
 -0.0038493092,
 0.02762856,
 -0.0016843395,
 0.0069401637,
 0.008839653,
 -0.028314

In [62]:
texts=["AI","MAchine learning","Deep Learning","Neural Network"]
batch_embeddings=embeddings.embed_documents(texts)
print(batch_embeddings[0])

[-0.0073426114, 0.012619104, -0.013854584, 0.0020128451, 0.01367185, 0.0039175483, 0.001127273, 0.025807675, -0.005160379, -0.053723533, -0.0102320835, 0.002724044, -0.027044877, -0.0130950175, 0.00858332, -0.016039109, 0.036129124, -0.0018698974, -0.00061798305, 0.01857806, -0.0049578287, 0.009287352, -0.001699126, 0.008612508, -0.017914047, 0.019682229, 0.0012921194, -0.008364114, -0.009403352, 0.12459677, -0.009583826, -0.014363433, 0.008546304, -0.0037073907, 0.014605985, -0.006372374, -0.006671376, -0.038708642, -0.0050331857, -0.0018501732, 0.0021430189, 0.009588995, 0.02053153, 0.008823104, -0.00057002465, 0.00039482498, -0.025550088, -0.0168815, 0.003978303, -0.0011011522, 0.015563737, 0.0128468685, -0.00601897, -0.021281403, 0.0063187755, 0.011781319, 0.012037379, -0.016949112, -0.00011030358, -0.01642362, -0.0101128025, 0.0016663779, 0.0207334, 0.038906798, -5.7802466e-05, 0.008486542, 0.012498707, -0.020971311, 0.0267337, -0.008236269, -0.0059556626, 0.019922422, -0.00244092

In [63]:
print(batch_embeddings[1])

[-0.026925143, -0.010011534, 0.0110168075, 0.016402202, -0.0008423096, 0.0022611853, -0.005512232, 0.009932523, 0.018647317, -0.061192226, -0.016538072, 0.011112397, -0.026245898, 0.005830677, -0.016184848, -0.016499167, 0.038908657, -0.009648675, 0.0050068223, -0.004801781, 0.0069189537, -0.010109284, 0.0045260377, 0.0012659893, -0.024905572, 0.014741623, -0.025793677, -0.0031963652, -0.014446643, 0.12992986, -0.022623196, -0.04110278, -0.0042664716, -0.009503045, 0.0033140806, -0.011698289, -0.0095273545, -0.013718701, -0.014340036, -0.010159077, -0.0042837276, 0.010780686, 0.028774505, 0.0076038283, 0.0031177881, -0.011303197, -0.019579314, -0.0016179061, 0.005258077, -0.018359149, -0.0025272528, 0.0051366175, 0.012373162, 0.0055191834, -0.0018112026, 0.00894055, 0.01760448, 0.020628603, 0.0064247902, -0.019188672, 0.0027985051, 0.011555601, 0.01933433, 0.026686128, 0.004566597, 0.0073052347, 0.00596913, -0.006669316, 0.0025515694, -0.0035523754, 0.027750388, 0.01975073, -0.00412017

In [64]:
### Compare Embedding using cosine similarity

def compare_embeddings(text1:str,text2:str):
    """Compare semantic simialrity of 2 texts usign embeddings"""

    emb1=np.array(embeddings.embed_query(text1))
    emb2=np.array(embeddings.embed_query(text2))

    ## Calculate the simialrity score

    similarity=np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2))
    return similarity

In [65]:
# Test semantic similarity
print("\nSemantic Similarity Examples:")
print(f"'AI' vs 'Artificial Intelligence': {compare_embeddings('AI', 'Artificial Intelligence'):.3f}")


Semantic Similarity Examples:
'AI' vs 'Artificial Intelligence': 0.813


In [66]:
print(f"'AI' vs 'Pizza': {compare_embeddings('AI', 'Pizza'):.3f}")

'AI' vs 'Pizza': 0.599


In [67]:
print(f"'Machine Learning' vs 'ML': {compare_embeddings('Machine Learning', 'ML'):.3f}")

'Machine Learning' vs 'ML': 0.632


### Create FAISS Vector Store

In [68]:
# !pip install faiss-cpu

In [69]:
vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)
print(f"Vector store created with {vectorstore.index.ntotal} vectors")

Vector store created with 4 vectors


In [70]:
vectorstore

In [71]:
## Save vector tore for later use
vectorstore.save_local("faiss_index")
print("Vector store saved to 'faiss_index' directory")

Vector store saved to 'faiss_index' directory


In [72]:
## load vector store
loaded_vectorstore=FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)

print(f"Loaded vector store contains {loaded_vectorstore.index.ntotal} vectors")

Loaded vector store contains 4 vectors


In [73]:
## Similarity Search
query="What is deep learning"

results=vectorstore.similarity_search(query,k=3)
print(results)

[Document(id='1b531eef-96ca-4bbb-9462-12d0268cb415', metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolutionized computer vision, NLP, and speech recognition.'), Document(id='8b07dfda-26e0-49d5-9aff-27516b8ff795', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'), Document(id='4840590e-a589-4581-8419-8fdfb8a9ffde', metadata={'source': 'NLP Overview', 'page': 1, 'topic': 'NLP'}, page_content='Natural Language Processing (NLP) is a branch of AI that helps computers understand human lang

In [74]:
print(f"Query: {query}\n")
print("Top 3 similar chunks:")
for i, doc in enumerate(results):
    print(f"\n{i+1}. Source: {doc.metadata['source']}")
    print(f"   Content: {doc.page_content[:200]}...")

Query: What is deep learning

Top 3 similar chunks:

1. Source: Deep Learning
   Content: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses multiple layers to progressively extract higher-level features from raw input.
        Deep learning ...

2. Source: ML Basics
   Content: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being explicitly programmed, ML algorithms find patterns in data.
        Common types include supervised...

3. Source: NLP Overview
   Content: Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
        It combines computational linguistics with machine learning and deep learning models.
      ...


In [75]:
### Similarity Search with score
results_with_scores=vectorstore.similarity_search_with_score(query,k=3)

print("\n\nSimilarity search with scores:")
for doc, score in results_with_scores:
    print(f"\nScore: {score:.3f}")
    print(f"Source: {doc.metadata['source']}")
    print(f"Content preview: {doc.page_content[:100]}...")



Similarity search with scores:

Score: 0.407
Source: Deep Learning
Content preview: Deep Learning is a subset of machine learning based on artificial neural networks.
        It uses m...

Score: 0.650
Source: ML Basics
Content preview: Machine Learning is a subset of AI that enables systems to learn from data.
        Instead of being...

Score: 0.687
Source: NLP Overview
Content preview: Natural Language Processing (NLP) is a branch of AI that helps computers understand human language.
...


In [76]:
chunks

[Document(metadata={'source': 'AI Introduction', 'page': 1, 'topic': 'AI'}, page_content='Artificial Intelligence (AI) is the simulation of human intelligence in machines.\n        These systems are designed to think like humans and mimic their actions.\n        AI can be categorized into narrow AI and general AI.'),
 Document(metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.'),
 Document(metadata={'source': 'Deep Learning', 'page': 1, 'topic': 'DL'}, page_content='Deep Learning is a subset of machine learning based on artificial neural networks.\n        It uses multiple layers to progressively extract higher-level features from raw input.\n        Deep learning has revolutionized computer vision, NLP, and speech recogn

In [77]:
### Search with metadata filtering
filter_dict={"topic":"ML"}
filtered_results=vectorstore.similarity_search(
    query,
    k=3,
    filter=filter_dict
)
print(filtered_results)

[Document(id='8b07dfda-26e0-49d5-9aff-27516b8ff795', metadata={'source': 'ML Basics', 'page': 1, 'topic': 'ML'}, page_content='Machine Learning is a subset of AI that enables systems to learn from data.\n        Instead of being explicitly programmed, ML algorithms find patterns in data.\n        Common types include supervised, unsupervised, and reinforcement learning.')]


In [78]:
len(filtered_results)

1

### Build RAG Chain With LCEL

In [79]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=userdata.get('GROQ_API_KEY'),
    temperature=0
)

In [80]:
llm.invoke("Hi")

AIMessage(content='Hello! How can I help you today?', additional_kwargs={'reasoning_content': 'The user says "Hi". We need to respond appropriately. It\'s a simple greeting. We can respond friendly.'}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 72, 'total_tokens': 113, 'completion_time': 0.085393548, 'completion_tokens_details': {'reasoning_tokens': 23}, 'prompt_time': 0.002875144, 'prompt_tokens_details': None, 'queue_time': 0.050499718, 'total_time': 0.088268692}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_0708ac49a5', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e9c9b-8a6c-7650-8e44-e00aec6bfd68-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 41, 'total_tokens': 113, 'output_token_details': {'reasoning': 23}})

In [81]:
# 1. Simple RAG Chain with LCEL
simple_prompt = ChatPromptTemplate.from_template("""Answer the question based only on the following context:
Context: {context}

Question: {question}

Answer:""")

In [82]:
## Basic retriever
retriever=vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":3}
)

In [83]:
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7ecb10c9e720>, search_kwargs={'k': 3})

In [84]:
from typing import List
# Format documents for the prompt
def format_docs(docs: List[Document]) -> str:
    """Format documents for insertion into prompt"""
    formatted = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get('source', 'Unknown')
        formatted.append(f"Document {i+1} (Source: {source}):\n{doc.page_content}")
    return "\n\n".join(formatted)

In [85]:
simple_rag_chain=(
    {"context":retriever | format_docs,"question":RunnablePassthrough() }
    | simple_prompt
    | llm
    |StrOutputParser()

)

In [86]:
simple_rag_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7ecb10c9e720>, search_kwargs={'k': 3})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\nContext: {context}\n\nQuestion: {question}\n\nAnswer:'), additional_kwargs={})])
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x7ecb10ca0050>,

In [87]:
### Conversational RAg Chain

conversational_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Use the provided context to answer questions."),
    ("placeholder", "{chat_history}"),
    ("human", "Context: {context}\n\nQuestion: {input}"),
])

In [88]:
def create_conversational_rag():
    """Create a conversational RAG chain with memory"""
    return (
        RunnablePassthrough.assign(
            context=lambda x: format_docs(retriever.invoke(x["input"]))
        )
        | conversational_prompt
        | llm
        | StrOutputParser()
    )

conversational_rag = create_conversational_rag()

In [89]:
conversational_rag

RunnableAssign(mapper={
  context: RunnableLambda(lambda x: format_docs(retriever.invoke(x['input'])))
})
| ChatPromptTemplate(input_variables=['context', 'input'], optional_variables=['chat_history'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag=

In [90]:
### streaming RAG chain
streaming_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | simple_prompt
    | llm
)

print("Modern RAG chains created successfully!")
print("Available chains:")
print("- simple_rag_chain: Basic Q&A")
print("- conversational_rag: Maintains conversation history")
print("- streaming_rag_chain: Supports token streaming")

Modern RAG chains created successfully!
Available chains:
- simple_rag_chain: Basic Q&A
- conversational_rag: Maintains conversation history
- streaming_rag_chain: Supports token streaming


In [91]:
# Test function for different chain types
def test_rag_chains(question: str):
    """Test all RAG chain variants"""
    print(f"Question: {question}")
    print("=" * 80)

    # 1. Simple RAG
    print("\n1. Simple RAG Chain:")
    answer = simple_rag_chain.invoke(question)
    print(f"Answer: {answer}")

    print("\n2. Streaming RAG:")
    print("Answer: ", end="", flush=True)
    for chunk in streaming_rag_chain.stream(question):
        print(chunk.content, end="", flush=True)
    print()

In [92]:
test_rag_chains("What is the difference between AI and machine learning")

Question: What is the difference between AI and machine learning

1. Simple RAG Chain:
Answer: AI (Artificial Intelligence) is the broad field that aims to simulate human intelligence in machines, enabling them to think, reason, and act like humans. It encompasses any approach that tries to replicate or mimic human cognitive abilities, and it is divided into categories such as narrow AI and general AI.

Machine Learning (ML) is a **subset of AI**. It specifically refers to techniques that allow systems to learn from data rather than being explicitly programmed. ML algorithms discover patterns in data and improve their performance over time, and they include methods like supervised, unsupervised, and reinforcement learning.

**In short:** AI is the overall discipline of creating intelligent behavior, while machine learning is a particular way of achieving AI—by training models on data to automatically learn patterns and make predictions.

2. Streaming RAG:
Answer: AI (Artificial Intelli

In [93]:
# Test with multiple questions
test_questions = [
    "What is the difference between AI and Machine Learning?",
    "Explain deep learning in simple terms",
    "How does NLP work?"
]

for question in test_questions:
    print("\n" + "=" * 80 + "\n")
    test_rag_chains(question)



Question: What is the difference between AI and Machine Learning?

1. Simple RAG Chain:
Answer: AI (Artificial Intelligence) is the broad field that aims to simulate human intelligence in machines, enabling them to think, reason, and act like humans. It encompasses any technique that gives a system intelligent behavior, ranging from rule‑based systems to advanced robotics, and is divided into narrow AI (focused on specific tasks) and general AI (human‑level intelligence).

Machine Learning (ML) is a specific subset of AI. Instead of being explicitly programmed with fixed rules, ML algorithms learn from data by finding patterns and building models that can make predictions or decisions. Supervised, unsupervised, and reinforcement learning are common types of ML.

**In short:** AI is the overall discipline of creating intelligent machines, while Machine Learning is a particular approach within AI that achieves intelligence by learning from data.

2. Streaming RAG:
Answer: AI (Artificia

In [94]:
## Conversational example
print("\n3. Conversational RAG Example:")
chat_history = []

# First question
q1 = "What is machine learning?"
a1 = conversational_rag.invoke({
    "input": q1,
    "chat_history": chat_history
})

print(f"Q1: {q1}")
print(f"A1: {a1}")


3. Conversational RAG Example:
Q1: What is machine learning?
A1: Machine learning (ML) is a branch of artificial intelligence that enables systems to learn from data rather than being explicitly programmed. By analyzing data, ML algorithms discover patterns and build models that can make predictions or decisions on new, unseen inputs. Common approaches include supervised learning, unsupervised learning, and reinforcement learning.


In [95]:
# Update history
chat_history.extend([
    HumanMessage(content=q1),
    AIMessage(content=a1)
])

In [96]:
# Follow-up question
q2 = "How is it different from traditional programming?"
a2 = conversational_rag.invoke({
    "input": q2,
    "chat_history": chat_history
})
print(f"\nQ2: {q2}")
print(f"A2: {a2}")


Q2: How is it different from traditional programming?
A2: **Machine learning vs. traditional (explicit) programming**

| Aspect | Traditional Programming | Machine‑Learning‑Based Approach |
|--------|------------------------|---------------------------------|
| **How behavior is created** | The developer writes **explicit rules** (if‑else, loops, formulas) that tell the computer exactly what to do for every possible input. | The developer writes a **learning algorithm** and supplies data; the system **discovers patterns** and builds a model that makes predictions or decisions automatically. |
| **Source of knowledge** | Knowledge is **hand‑crafted** by the programmer (domain expertise encoded as code). | Knowledge is **extracted from data**; the model’s “knowledge” is the statistical relationships it has learned. |
| **Adaptability** | Changing behavior requires **modifying the code** and redeploying it. | The system can **re‑train** on new data to adapt to changing conditions without